# Eye Movement-Based Schizophrenia Recognition — Full Pipeline

| Cell | Mục đích |
|---|---|
| 1 | Mount Drive + cd vào project |
| 2 | 🔴 XÓA kết quả cũ (bỏ comment khi cần reset) |
| 3 | Install thư viện còn thiếu |
| 4 | Tier 1 — Preprocessing |
| 5 | Tier 2 — Feature Engineering |
| 6 | Tier 3 — Tabular (XGBoost) |
| 7 | Tier 4A — ResNet50 extraction |
| 8 | Tier 4B — Build graphs |
| 9 | Tier 4C — GNN-CEFAM training |
| 10 | Tier 4D — BiCA-HS training |
| 11 | Tier 5 — Meta-Learner |
| 12 | Tổng hợp kết quả |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition"

In [ ]:
# 🔴 XÓA KẾT QUẢ CŨ — bỏ comment từng dòng tùy mức độ reset

# Xóa checkpoint + OOF Tier 4 (BẮT BUỘC sau khi fix C-1, C-2, C-3)
# !rm -rf results/bica/ results/cefam/ results/stgnn/ results/tier5/
# !rm -rf "Bidirectional Cross-Attention Hybrid Stream/results/checkpoints/"

# Xóa Tier 3
# !rm -rf results/baselines/

# Xóa graphs (rebuild từ đầu)
# !rm -rf data/processed/graphs/

# Xóa toàn bộ (chạy lại từ raw data)
# !rm -rf results/ data/processed/ data/external/

In [ ]:
# Colab đã có sẵn torch/sklearn/pandas — chỉ install thêm cái còn thiếu
!pip install torch-geometric
!pip install pyyaml omegaconf optuna catboost pyarrow openpyxl xgboost lightgbm shap

In [ ]:
# Tier 1: Tạo category map + tiền xử lý dữ liệu thô
!python -m src.utils.generate_category_map
!python -m src.tier1_preprocessing.preprocess

In [ ]:
# Tier 2: Trích xuất đặc trưng stimulus-level + subject-level delta
!python -m src.tier2_features.stimulus_features
!python -m src.tier2_features.subject_aggregator

In [ ]:
# Tier 3: Tabular baseline (XGBoost)
!python -m src.tier3_tabular.tabular_models --model xgboost

In [ ]:
# Tier 4A: Trích xuất ResNet50 visual features (2048-dim)
# Output: data/external/feature_dict_ResNet50.npy
!python -m src.utils.extract_resnet_features

In [ ]:
# Tier 4B: Xây dựng đồ thị PyG từ fixation data + ResNet50 features
# Output: data/processed/graphs/graphs.pt
!python -m src.tier4_advanced.graph_builder

In [ ]:
# Tier 4C: Huấn luyện GNN-CEFAM (4-fold GroupKFold)
# Output: results/cefam/cefam_oof_subject_preds.csv
!python scripts/train_tier4.py

In [ ]:
# Tier 4D: Huấn luyện BiCA-HS (4-fold GroupKFold)
# PYTHONPATH=. bắt buộc để import src.*
# Output: results/bica/bica_subject_val_predictions.csv
!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" --config configs/bica_config.yaml

In [ ]:
# Tier 5: Meta-Learner kết hợp Tier 3 + Tier 4
# Mặc định dùng XGBoost OOF + CEFAM OOF
!python scripts/run_tier5.py --plot --calibrate

In [ ]:
# Tier 5 (phiên bản dùng BiCA-HS OOF thay vì CEFAM)
!python scripts/run_tier5.py \
    --tier4-oof results/bica/bica_subject_val_predictions.csv \
    --plot --calibrate

In [ ]:
# Tổng hợp kết quả
import json, os
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

def show(path, name):
    if not os.path.exists(path):
        return
    df = pd.read_csv(path)
    if 'Label' not in df.columns or 'Pred_Proba_Subject' not in df.columns:
        return
    y, p = df['Label'].values, df['Pred_Proba_Subject'].values
    print(f"{name:<35} AUC={roc_auc_score(y,p):.4f}  ACC={accuracy_score(y,(p>=.5).astype(int)):.4f}  F1={f1_score(y,(p>=.5).astype(int)):.4f}")

print("=" * 70)
show("results/baselines/xgboost_oof_subject_preds.csv",  "Tier 3 XGBoost")
show("results/cefam/cefam_oof_subject_preds.csv",        "Tier 4 GNN-CEFAM")
show("results/bica/bica_subject_val_predictions.csv",    "Tier 4 BiCA-HS")

t5 = "results/tier5/tier5_results_summary.json"
if os.path.exists(t5):
    d = json.load(open(t5))
    if "best_model" in d:
        b = d["best_model"]
        print(f"{'Tier 5 Meta-Learner':<35} AUC={b.get('auc',0):.4f}  ACC={b.get('acc',0):.4f}  F1={b.get('f1',0):.4f}")
print("=" * 70)
print("SOTA (MSNet, IEEE TNNLS 2025):      AUC=0.8854  ACC=0.8125")